# Sprint 4 - Tópico 1 e Tópico 2: Engenharia de Atributos e Seleção Parcimoniosa

**Projeto:** Previsão de Risco de Baixa Produtividade Agrícola (Soja em Sorriso - MT)  
**Trilha:** C — Risco de baixa produtividade agrícola  
**Responsável (Cards Trello):** João Guilherme Cravo (JS)  
**Objetivos:**
- **TÓPICO 1:** Criar variáveis climáticas inéditas focadas nos erros da Sprint 3 (especificamente o falso negativo da safra 2021) sem vazamento temporal.
- **TÓPICO 2:** Realizar seleção parcimoniosa de atributos compatível com $N=8$ no treino (2015–2022), avaliando correlação e multicolinearidade para entregar o subconjunto enxuto aos tópicos seguintes (Jean e Samuel).


In [1]:
%pip install -r ../requirements.txt


In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Ambiente e bibliotecas carregados com sucesso!')


## Tópico 1: Engenharia de Novas Features Focadas nos Erros da S3

### Diagnóstico do Problema da Sprint 3
Na Sprint 3, os modelos receberam apenas atributos climáticos acumulados anualmente.
A **safra 2021** sofreu quebra severa de produtividade (3.360 kg/ha < limiar de 3.400 kg/ha), porém teve chuva anual de 1.634,39 mm (o 3º maior volume da década).
O modelo gerou um **Falso Negativo (FN)** crítico para 2021 porque o acumulado anual mascarou estresses severos ocorridos durante a formação dos grãos.

### Respaldo Agronômico e Janela Crítica
Em Sorriso-MT, a semeadura ocorre entre outubro e novembro, e o período crítico de **floração e enchimento de grãos** concentra-se no bimestre **janeiro e fevereiro**.
Extraímos dos dados diários da NASA POWER (`data/raw/nasa_power_Sorriso_MT_2015_2025.json`) as seguintes variáveis para esse período:
1. `deficit_hidrico_solo`: Contagem de dias em Jan-Fev com `GWETROOT < 0.70` (déficit hídrico radicular crítico).
2. `dias_pico_calor`: Contagem de dias em Jan-Fev com `T2M_MAX >= 35.0 °C` (estresse térmico que inviabiliza pólen e causa abortamento floral/vagens).
3. `veranico_max_dias`: Maior sequência ininterrupta de dias com chuva $< 1.0$ mm em Jan-Fev.
4. `precipitacao_enchimento`: Volume acumulado de precipitação estritamente em Jan-Fev (mm).

*Garantia Anti-Vazamento:* O cômputo encerra-se no último dia de fevereiro, 1 a 2 meses antes da colheita e do fechamento oficial dos dados do IBGE PAM.


In [3]:
# 1. Carregar dados climáticos diários brutos da NASA POWER
with open('../data/raw/nasa_power_Sorriso_MT_2015_2025.json') as f:
    raw_nasa = json.load(f)

params = raw_nasa['properties']['parameter']
datas = sorted(params['PRECTOTCORR'].keys())

registros_diarios = []
for dt in datas:
    registros_diarios.append({
        'data': dt,
        'ano': int(dt[:4]),
        'mes': int(dt[4:6]),
        'prec': float(params['PRECTOTCORR'][dt]),
        't2m_max': float(params['T2M_MAX'][dt]),
        'gwetroot': float(params['GWETROOT'][dt])
    })

df_diario = pd.DataFrame(registros_diarios)
df_diario = df_diario.replace(-999.0, np.nan)

# 2. Filtrar apenas safras consolidadas (2015 a 2024) e bimestre crítico (Janeiro e Fevereiro)
df_jf = df_diario[(df_diario['ano'] >= 2015) & (df_diario['ano'] <= 2024) & (df_diario['mes'].isin([1, 2]))].copy()

# 3. Calcular novas métricas por safra
novas_features = []
for ano, grupo in df_jf.groupby('ano'):
    dias_calor = int((grupo['t2m_max'] >= 35.0).sum())
    dias_solo_seco = int((grupo['gwetroot'] < 0.70).sum())
    chuva_jf = float(grupo['prec'].sum())
    
    # Maior veranico consecutivo (chuva < 1.0 mm)
    is_dry = (grupo['prec'] < 1.0).astype(int)
    max_veranico = 0
    cur_v = 0
    for v in is_dry:
        if v == 1:
            cur_v += 1
            if cur_v > max_veranico:
                max_veranico = cur_v
        else:
            cur_v = 0
            
    novas_features.append({
        'ano': ano,
        'dias_pico_calor': dias_calor,
        'deficit_hidrico_solo': dias_solo_seco,
        'veranico_max_dias': max_veranico,
        'precipitacao_enchimento': round(chuva_jf, 2)
    })

df_novas = pd.DataFrame(novas_features)
display(df_novas)


In [4]:
# Integrar com o dataset processed existente
df_existente = pd.read_csv('../data/processed/dados_modelo.csv')

# Manter colunas originais e atualizar com as novas
colunas_originais = [
    'ano', 'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT',
    'quebra_safra', 'amplitude_termica', 'precipitacao_por_temperatura', 'GWET_x_PREC'
]
df_base = df_existente[[c for c in colunas_originais if c in df_existente.columns]].drop_duplicates(subset=['ano'])

df_integrado = pd.merge(df_base, df_novas, on='ano', how='inner')
display(df_integrado.head())
print(f'Dataset consolidado: {len(df_integrado)} safras (2015-2024) e {len(df_integrado.columns)} colunas.')


## Tópico 2: Seleção Parcimoniosa de Atributos (Compatível com $N=8$)

### Objetivo e Cuidados Metodológicos
Com tamanho amostral reduzido no bloco de treino ($N_{train} = 8$, safras 2015 a 2022), o uso de muitos regressores correlacionados gera superajuste (*overfitting*).
- **Regra anti-vazamento:** A avaliação de correlação e multicolinearidade é realizada **estritamente no bloco de treino (2015 a 2022)**. Nenhuma linha de teste (2023–2024) entra na seleção.
- **Critério de corte:** Selecionar um subconjunto enxuto de 3 a 5 variáveis com alta correlação com o alvo e baixa redundância mútua.


In [5]:
target = 'quebra_safra'

# Isolar bloco de treino (2015-2022)
train_mask = df_integrado['ano'] < 2023
df_train = df_integrado[train_mask].copy()

features_candidatas = [
    'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT',
    'amplitude_termica', 'precipitacao_por_temperatura', 'GWET_x_PREC',
    'dias_pico_calor', 'deficit_hidrico_solo', 'veranico_max_dias', 'precipitacao_enchimento'
]

# Correlação com o alvo exclusivamente no treino
corr_treino = df_train[features_candidatas + [target]].corr()[target].sort_values(ascending=False)

print('=== 1. CORRELAÇÃO LINEAR COM O ALVO NO TREINO (N=8) ===')
display(corr_treino.to_frame(name='Correlação com quebra_safra'))


In [6]:
# Matriz de multicolinearidade entre as principais candidatas
top_features = ['deficit_hidrico_solo', 'dias_pico_calor', 'precipitacao_enchimento', 'amplitude_termica', 'GWETROOT', 'PRECTOTCORR']

plt.figure(figsize=(8, 6))
sns.heatmap(df_train[top_features].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Multicolinearidade no Bloco de Treino (2015–2022)', fontweight='bold')
plt.tight_layout()
plt.show()


### Justificativa de Descarte de Redundâncias e Seleção Final

1. **Variáveis Descartadas:**
   - `PRECTOTCORR`, `GWETROOT` anual e `GWET_x_PREC`: Apresentam multicolinearidade severa mútua ($r > 0.93$) e correlação mais fraca com quebra de safra, além de mascararem o veranico ocorrido em 2021.
   - `T2M_MAX` e `T2M_MIN` brutas anuais: Redundantes com `amplitude_termica`.
   - `precipitacao_por_temperatura`: Redundante com `precipitacao_enchimento`.

2. **Lista Final Selecionada (4 Atributos Parcimoniosos):**
   - `deficit_hidrico_solo`: Correlação altíssima ($r = +0.9027$) com a quebra real, identificando com precisão os 42 dias de seca na raiz em 2021 e 47 em 2015.
   - `dias_pico_calor`: Correlação positiva forte ($r = +0.7354$), capturando o estresse térmico reprodutivo.
   - `amplitude_termica`: Preservada da Sprint 2 para oscilação térmica.
   - `precipitacao_enchimento`: Correlação negativa ($r = -0.5545$), balanço hídrico intra-safra sem colinearidade com chuva anual.


In [7]:
features_finais = [
    'deficit_hidrico_solo',
    'dias_pico_calor',
    'amplitude_termica',
    'precipitacao_enchimento'
]

print('=== LISTA FINAL DE ATRIBUTOS SELECIONADOS PARA O MODELO ===')
for i, f in enumerate(features_finais, 1):
    print(f'{i}. {f} (r={corr_treino[f]:+.4f})')

# Salvar o dataset atualizado na camada processed para que Jean (Tópico 3) e Samuel (Tópico 5) consumam
path_processed = '../data/processed/dados_modelo.csv'
df_integrado.to_csv(path_processed, index=False)
print(f'\nDataset salvo com sucesso em: {path_processed}')
print(f'Total de registros: {len(df_integrado)} safras.')
